In [ ]:
import torch
from torch import nn
from torchview import draw_graph

class VectorQuantizer(nn.Module):
    def __init__(self, codebook_entries=1024, dim=16):
        super().__init__()

        self.emb = nn.Embedding(num_embeddings=codebook_entries, embedding_dim=dim)
        nn.init.uniform_(self.emb.weight, 0.0, 1.0)

    def forward(self, x, efficient=True):
        if not efficient:
            batch_size = x.shape[0]
            emb = self.emb.weight.repeat(batch_size, 1, 1)
            x = x.unsqueeze(1)
            distences = torch.sum((x - emb) ** 2, dim=-1)  # [B, N, D] -> [B, N]
        else:
            # L**2 + C**2 - 2LC
            L2 = torch.sum(self.emb.weight**2, dim=-1).unsqueeze(0)  # [1, N]
            C2 = torch.sum(x**2, dim=-1, keepdim=True)               # [B, 1]
            LC = x @ self.emb.weight.t()                             # [B, N]

            distences = L2 - 2 * LC + C2

        min_distence = torch.argmin(distences, dim=-1)
        return self.emb.weight[min_distence]


class VectorQuantizer_VariaonalAutoEncoder(nn.Module):
    def __init__(self, dim = 1024,  codebook_entries = 64, latent_dim = 256):
        super().__init__()
        
        self.enc = nn.Linear(dim, latent_dim)
        self.dec = nn.Linear(latent_dim, dim)
        self.vq  = VectorQuantizer(codebook_entries, latent_dim)
        
    def forward(self, x):
        z = self.enc(x)
        e = self.vq(z)
        return self.dec(e)
    
vqvae = VectorQuantizer_VariaonalAutoEncoder(4, 6, 2)
rand = torch.rand((8, 4))
print(vqvae(rand))

mg = draw_graph(vqvae, rand)
mg.visual_graph